In [0]:
-- Transform Bronze to Silver: Cast types, filter out bad/null records, and deduplicate keeping the latest ingested record per movie.
CREATE OR REPLACE TABLE workspace.mini_etl.silver_movies AS
SELECT
    CAST(movie_id AS INT)  AS movie_id,
    TRIM(title)            AS title,
    CAST(year AS INT)       AS release_year,
    TRIM(genre)             AS genre,
    CAST(rating AS DECIMAL(3,1))    AS rating,
    ingested_at             AS bronze_ingested_at,
    current_timestamp()     AS processed_at
FROM workspace.mini_etl.bronze_movies
WHERE _rescued_data IS NULL
    AND movie_id IS NOT NULL
    AND title IS NOT NULL
    AND rating BETWEEN 0 AND 10
    AND year BETWEEN 1888 AND year(current_date())
QUALIFY ROW_NUMBER() OVER (PARTITION BY movie_id ORDER BY ingested_at DESC) = 1;

In [0]:
-- Validation Check: Compare row counts between Bronze (raw) and Silver (cleaned) layers.
-- Silver should be <= Bronze due to filtering and deduplication.
SELECT
  (SELECT COUNT(*) FROM workspace.mini_etl.bronze_movies) AS bronze_rows,
  (SELECT COUNT(*) FROM workspace.mini_etl.silver_movies) AS silver_rows;